# Primary Notebook for Collaborative Filtering experimenting

### 0. Words

### 1. Imports and setup

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

### 2. Loading data

In [4]:
def load_ratings(ratings_csv: str):
    """
    Loads the ratings from CSV and returns a pandas DataFrame
    containing columns [userId, movieId, rating].
    """
    df = pd.read_csv(ratings_csv)
    # Ensure the correct columns exist
    df = df[['userId', 'movieId', 'rating']]
    return df

### 3. Constructing utility matrix

In [5]:
def build_utility_matrix(train_df: pd.DataFrame):
    """
    Builds a user-item rating matrix (utility matrix) from the training DataFrame.

    Returns:
       - utility_matrix (num_users x num_items) filled with np.nan where no rating is present
       - user_ids (mapping index -> userId)
       - item_ids (mapping index -> movieId)
    """
    user_ids = train_df['userId'].unique()
    item_ids = train_df['movieId'].unique()

    # Sort them so indices remain consistent
    user_ids = np.sort(user_ids)
    item_ids = np.sort(item_ids)

    # Map from userId/movieId to row/col indices
    user_index = {u: idx for idx, u in enumerate(user_ids)}
    item_index = {i: idx for idx, i in enumerate(item_ids)}

    # Initialize matrix with np.nan
    utility_matrix = np.full((len(user_ids), len(item_ids)), np.nan, dtype=np.float32)

    # Fill the matrix with known ratings
    for row in train_df.itertuples():
        u = row.userId
        i = row.movieId
        r = row.rating
        utility_matrix[user_index[u], item_index[i]] = r

    return utility_matrix, user_ids, item_ids

### 4. Similarity computations

In [6]:
def cosine_similarity(matrix: np.ndarray, epsilon=1e-9):
    """
    Computes cosine similarity on rows of a matrix.
    matrix shape: (num_entities x num_features)

    Returns a similarity matrix of shape (num_entities x num_entities).
    """
    # Replace NaN with 0 for similarity computation
    mat_filled = np.nan_to_num(matrix, nan=0.0)

    # Norm for each row
    norms = np.sqrt((mat_filled**2).sum(axis=1, keepdims=True)) + epsilon

    # Compute dot product
    dot_products = mat_filled @ mat_filled.T

    # Outer product of norms for scaling
    norm_matrix = norms @ norms.T

    sim_matrix = dot_products / norm_matrix
    # Clip potential numerical errors outside [0,1]
    sim_matrix = np.clip(sim_matrix, 0, 1)
    return sim_matrix

def pearson_correlation(matrix: np.ndarray, epsilon=1e-9):
    """
    Computes Pearson correlation on rows of a matrix, ignoring NaNs in each row's mean.

    Returns a similarity matrix with values in [-1,1].
    """
    # Center each row by subtracting the mean (ignoring NaNs)
    mean_vals = np.nanmean(matrix, axis=1, keepdims=True)
    centered_matrix = matrix - mean_vals
    # Replace NaNs with 0
    centered_matrix = np.nan_to_num(centered_matrix, nan=0.0)

    # Dot product
    dot_products = centered_matrix @ centered_matrix.T
    # Norm of centered rows
    norms = np.sqrt((centered_matrix**2).sum(axis=1, keepdims=True)) + epsilon
    norm_matrix = norms @ norms.T

    corr_matrix = dot_products / norm_matrix
    # Clip to ensure numerical stability
    corr_matrix = np.clip(corr_matrix, -1, 1)
    return corr_matrix

### 5. Prediction

In [11]:
def predict_rating_user_based(user_idx, item_idx, utility_matrix, user_sim_matrix, k=20):
    """
    Predict the rating of a target user for a specific item 
    using User-Based Collaborative Filtering with top-k neighbors.

    Steps:
    1) Identify users who rated the target item.
    2) Sort them by similarity to the target user.
    3) Take top-k neighbors and compute a weighted average (using similarity).
    4) Incorporate mean-centering if desired (classic approach).
    """
    # Rating by target user (row = user_idx)
    target_user_ratings = utility_matrix[user_idx, :]

    # List of users who have rated this item
    users_who_rated_item = np.where(~np.isnan(utility_matrix[:, item_idx]))[0]
    if len(users_who_rated_item) == 0:
        # No one rated this item -> fallback to a default, e.g., global mean
        return np.nan  # or a global average rating

    # Similarities to the target user
    sims = user_sim_matrix[user_idx, users_who_rated_item]

    # Sort neighbors by similarity (descending)
    sorted_neighbors = users_who_rated_item[np.argsort(-sims)]
    # Take top-k
    top_k_neighbors = sorted_neighbors[:k]
    top_k_sims = sims[np.argsort(-sims)][:k]

    # Mean rating of the target user
    target_user_mean = np.nanmean(target_user_ratings)

    # Weighted sum
    numerator = 0.0
    denominator = 0.0

    for idx_neighbor, sim in zip(top_k_neighbors, top_k_sims):
        # rating from neighbor
        neighbor_rating = utility_matrix[idx_neighbor, item_idx]
        # mean rating of the neighbor
        neighbor_mean = np.nanmean(utility_matrix[idx_neighbor, :])

        # Weighted deviation from neighbor's mean
        numerator += sim * (neighbor_rating - neighbor_mean)
        denominator += abs(sim)

    if denominator == 0.0:
        # fallback to user mean if no neighbors or zero-sum similarity
        return target_user_mean

    predicted = target_user_mean + (numerator / denominator)
    return predicted

def predict_rating_item_based(user_idx, item_idx, utility_matrix, item_sim_matrix, k=20):
    """
    Predict the rating of a target user for a specific item 
    using Item-Based Collaborative Filtering with top-k similar items.
    """
    # Ratings for the target item across all users
    target_item_column = utility_matrix[:, item_idx]

    # Which items has user_idx actually rated?
    items_rated_by_user = np.where(~np.isnan(utility_matrix[user_idx, :]))[0]
    if len(items_rated_by_user) == 0:
        # user hasn't rated anything -> fallback
        return np.nan

    # Similarities for the target item to other items
    sims = item_sim_matrix[item_idx, items_rated_by_user]

    # Sort neighbors by similarity
    sorted_neighbors = items_rated_by_user[np.argsort(-sims)]
    # Take top-k
    top_k_neighbors = sorted_neighbors[:k]
    top_k_sims = sims[np.argsort(-sims)][:k]

    # Weighted sum
    numerator = 0.0
    denominator = 0.0

    for idx_neighbor, sim in zip(top_k_neighbors, top_k_sims):
        neighbor_rating = utility_matrix[user_idx, idx_neighbor]
        numerator += sim * neighbor_rating
        denominator += abs(sim)

    if denominator == 0.0:
        return np.nan  # fallback: no similar items or zero similarity

    predicted = numerator / denominator
    return predicted

### 6. Evaluation (RMSE & MAE)

In [13]:
def evaluate(test_df, utility_matrix, user_ids, item_ids, sim_matrix, 
             prediction_func, user_based=True, k=20):
    """
    Evaluate predictions on a test set and return RMSE and MAE.

    :param test_df: DataFrame with columns [userId, movieId, rating].
    :param utility_matrix: 2D array of shape (num_users, num_items).
    :param user_ids: array of unique userIds in sorted order.
    :param item_ids: array of unique itemIds in sorted order.
    :param sim_matrix: precomputed similarity matrix (user-user or item-item).
    :param prediction_func: function to compute rating predictions (user-based or item-based).
    :param user_based: bool, indicates if user_ids or item_ids are the 'rows' of the sim_matrix.
    :param k: int, number of neighbors to consider.

    :return: (rmse, mae)
    """
    predictions = []
    truths = []

    # For each (user,movie,rating) in the test set
    for row in test_df.itertuples():
        user_id = row.userId
        item_id = row.movieId
        true_rating = row.rating

        # If user/item not in training matrix, skip or fallback
        if user_id not in user_ids or item_id not in item_ids:
            continue

        user_idx = np.where(user_ids == user_id)[0][0]
        item_idx = np.where(item_ids == item_id)[0][0]

        # Predict
        pred_rating = prediction_func(user_idx, item_idx, utility_matrix, sim_matrix, k=k)
        if np.isnan(pred_rating):
            # fallback to global mean or skip
            pred_rating = np.nanmean(utility_matrix)

        predictions.append(pred_rating)
        truths.append(true_rating)

    predictions = np.array(predictions)
    truths = np.array(truths)

    # Compute RMSE
    rmse = np.sqrt(np.mean((predictions - truths)**2))
    # Compute MAE
    mae = np.mean(np.abs(predictions - truths))

    return rmse, mae

### 7. Main Experiment

In [17]:
def main():
    # ----------------------------------
    # Step A: Load and Split the Data
    # ----------------------------------
    ratings_csv = "ml-latest-small/ratings.csv"
    df = load_ratings(ratings_csv)

    # train_test_split from scikit-learn (80/20 split)
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

    print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")

    # -------------------------------------------------
    # Step B: Build utility matrix from training set
    # -------------------------------------------------
    utility_matrix, user_ids, item_ids = build_utility_matrix(train_df)

    # -----------------------------------------
    # Step C: Compute Similarities (User or Item)
    # -----------------------------------------
    # For user-based, compute user-user similarity
    user_cosine_sim = cosine_similarity(utility_matrix)              # shape: (num_users x num_users)
    user_pearson_sim = pearson_correlation(utility_matrix)           # shape: (num_users x num_users)

    # For item-based, compute item-item similarity
    # Transpose so items are rows => (num_items x num_users)
    # We'll just pass the transposed matrix to the same functions
    utility_matrix_items = utility_matrix.T  # shape: (num_items x num_users)
    item_cosine_sim = cosine_similarity(utility_matrix_items)        # shape: (num_items x num_items)
    item_pearson_sim = pearson_correlation(utility_matrix_items)     # shape: (num_items x num_items)

    # -----------------------------------------
    # Step D: Evaluate each approach
    # -----------------------------------------
    k_neighbors = 20  # you can experiment with different k

    # -- 1) User-Based, Cosine
    rmse_uc, mae_uc = evaluate(
        test_df, utility_matrix, user_ids, item_ids, 
        user_cosine_sim, predict_rating_user_based, 
        user_based=True, k=k_neighbors
    )
    print(f"User-Based, Cosine:    RMSE={rmse_uc:.4f}, MAE={mae_uc:.4f}")

    # -- 2) User-Based, Pearson
    rmse_up, mae_up = evaluate(
        test_df, utility_matrix, user_ids, item_ids, 
        user_pearson_sim, predict_rating_user_based, 
        user_based=True, k=k_neighbors
    )
    print(f"User-Based, Pearson:   RMSE={rmse_up:.4f}, MAE={mae_up:.4f}")

    # -- 3) Item-Based, Cosine
    rmse_ic, mae_ic = evaluate(
        test_df, utility_matrix, user_ids, item_ids, 
        item_cosine_sim, predict_rating_item_based, 
        user_based=False, k=k_neighbors
    )
    print(f"Item-Based, Cosine:    RMSE={rmse_ic:.4f}, MAE={mae_ic:.4f}")

    # -- 4) Item-Based, Pearson
    rmse_ip, mae_ip = evaluate(
        test_df, utility_matrix, user_ids, item_ids, 
        item_pearson_sim, predict_rating_item_based, 
        user_based=False, k=k_neighbors
    )
    print(f"Item-Based, Pearson:   RMSE={rmse_ip:.4f}, MAE={mae_ip:.4f}")


if __name__ == "__main__":
    main()

Train size: 80668, Test size: 20168
User-Based, Cosine:    RMSE=0.8933, MAE=0.6767
User-Based, Pearson:   RMSE=0.8859, MAE=0.6701
Item-Based, Cosine:    RMSE=0.8613, MAE=0.6540
Item-Based, Pearson:   RMSE=0.9511, MAE=0.7116
